In [ ]:
pip install torch transformers

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

if torch.cuda.is_available(): #Use mps for Mac
    device = "cuda"
else:
    device = "cpu"

print(f"Loading model on: {device} (first run downloads it, be patient)...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)
model.to(device)
model.eval()

Loading model on: cuda (first run downloads it, be patient)...


In [2]:
PERSONAS = {
    "casual": "You are a laid-back, friendly chatbot. Keep answers short and casual.",
    "tutor": (
        "You are a patient Python tutor. The student is a beginner. "
        "Explain simply and use small code examples."
    ),
    "finance": (
        "You explain finance concepts (banking, interest, investing basics) in plain English. "
        "You are not a financial advisor."
    ),
}

MAX_MESSAGES_KEPT = 12

def new_conversation(persona_name):
    return [{"role": "system", "content": PERSONAS[persona_name]}]

def chat(history, user_text):
    history.append({"role": "user", "content": user_text})
    if len(history) > MAX_MESSAGES_KEPT + 1:
        history[:] = [history[0]] + history[-MAX_MESSAGES_KEPT:]
    prompt = tokenizer.apply_chat_template(
        history, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    history.append({"role": "assistant", "content": reply})
    return reply

In [3]:
print("Chatbot ready! Commands:")
print("  /persona <name>  -> switch personality: casual, tutor, finance")
print("  /reset           -> forget the conversation")
print("  /exit            -> quit\n")

persona = "casual"
history = new_conversation(persona)

queries = [
    "Teach me python what is [::-2]",
    "Does [:] when assigned to a new variable make a copy of the list?",
    "What does %d mean in regex in python. Can you list all regex common syntax?",
    "Thank you, if I have any other Python questions I will ask you later. Can you update me on current events?",
    "What all stuff are you able to answer?",
    "Explain why waterfalls flow endlessly.",
    "Goodbye",
    "Nope"
]

for q in queries:
    print(f"You: {q}")
    reply = chat(history, q)
    print(f"Chatbot: {reply}\n")

Chatbot ready! Commands:
  /persona <name>  -> switch personality: casual, tutor, finance
  /reset           -> forget the conversation
  /exit            -> quit



You: Teach me python what is [::-2]
Chatbot: Sure! In Python, `[::-2]` means "start at the end of the list and move backwards by two elements". It's like slicing a list from the second last element to the first, but in reverse order.

You: Does [:] when assigned to a new variable make a copy of the list?
Chatbot: Yes, [:] assigns the entire list to a new variable. When you assign it to a new variable, it creates a shallow copy of the original list, meaning any changes made to the new variable won't affect the original list. If you want to create a deep copy (a copy that doesn’t change), you should use the `copy` module or manually iterate over the list multiple times to avoid affecting the original one.

You: What does %d mean in regex in python. Can you list all regex common syntax?
Chatbot: In regular expressions, `%d` stands for `digit`. This tells the regular expression engine to match exactly one digit (`0-9`) within each line of text.

You: Thank you, if I have any other Python q